<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2013%20-%202026-05-11%20-%20M%C3%A9tricas%20y%20comparaci%C3%B3n%20de%20modelos%20-%20ENTREGA%20SEMANA%203/clase_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 13 · Métricas y comparación de modelos · ENTREGA SEMANA 3

**Fecha:** Lunes 11 de mayo de 2026  
**Módulo:** Módulo 3 · Modelado y estadística inferencial  

---

## Introducción

Ya tenemos un modelo baseline, hoy vamos a medirlo bien: matriz de confusión, precisión, recall, F1, ROC-AUC. Compararemos 2 o 3 modelos con validación cruzada.

## 1. Matriz de confusión · la estructura base

<img src="https://upload.wikimedia.org/wikipedia/commons/2/26/Precisionrecall.svg" width="380" alt="Precision y Recall">

La **matriz de confusión** cruza la predicción con la realidad:

| | Predicho 0 | Predicho 1 |
|---|---|---|
| **Real 0** | TN (True Negative) | FP (False Positive) |
| **Real 1** | FN (False Negative) | TP (True Positive) |

A partir de estos 4 números se derivan todas las métricas:

- **Accuracy** = (TP + TN) / total. Porcentaje correcto.
- **Precision** = TP / (TP + FP). De los que predije positivos, ¿cuántos lo son de verdad?
- **Recall (sensibilidad)** = TP / (TP + FN). De los positivos reales, ¿cuántos detecté?
- **F1** = media armónica de precision y recall. Balance entre las dos.
- **Specificity** = TN / (TN + FP). De los negativos reales, ¿cuántos descarté?

## 2. ¿Qué métrica importa? · depende del caso

**Detección de fraude / cáncer**:
- Prioridad: **Recall alto**. Perder un caso positivo es muy caro.
- Precision puede ser menor: investigar falsos positivos es molesto pero manejable.

**Spam / filtros**:
- Prioridad: **Precision alta**. Bloquear un correo legítimo es caro.
- Recall puede ceder: dejar pasar algo de spam es tolerable.

**Balanceado (no hay asimetría clara)**:
- **F1** o **Accuracy**.

Si el dataset está **muy desbalanceado** (por ejemplo fraude en 1% de transacciones), accuracy engaña: un modelo que predice "no fraude" a todo tiene 99% de accuracy y 0% de utilidad. Ahí hay que mirar precision/recall.

## 3. La curva ROC · historia en WWII

<img src="https://upload.wikimedia.org/wikipedia/commons/6/6b/Roccurves.png" width="400" alt="ROC curves">

La curva **ROC** (*Receiver Operating Characteristic*) viene de la Segunda Guerra Mundial: ingenieros de radar intentaban distinguir aviones enemigos del ruido. La curva representa la relación entre **Recall** (tasa de positivos bien detectados) y **Tasa de falsos positivos** al variar el umbral de decisión.

El área bajo la curva (**AUC-ROC**) es un resumen entre 0.5 (aleatorio) y 1 (perfecto). Interpretación: es la probabilidad de que el modelo asigne una mayor puntuación a un positivo aleatorio que a un negativo aleatorio.

Ventaja: no depende del umbral. Desventaja: puede ser engañosa con clases muy desbalanceadas; en ese caso conviene usar **Precision-Recall curve** y su AUC (AP).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url).dropna(subset=["Age"]).copy()
df["SexN"] = (df["Sex"] == "female").astype(int)
X = df[["Age", "Pclass", "SibSp", "Parch", "SexN", "Fare"]]
y = df["Survived"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
pred = clf.predict(X_te)
proba = clf.predict_proba(X_te)[:, 1]

cm = confusion_matrix(y_te, pred)
ConfusionMatrixDisplay(cm, display_labels=["No", "Sí"]).plot()
plt.title("Matriz de confusión"); plt.show()

print(f"Accuracy : {accuracy_score(y_te, pred):.3f}")
print(f"Precision: {precision_score(y_te, pred):.3f}")
print(f"Recall   : {recall_score(y_te, pred):.3f}")
print(f"F1       : {f1_score(y_te, pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_te, proba):.3f}")

## 4. Validación cruzada · dejar de depender de una partición

Un solo `train_test_split` da una estimación ruidosa: si partieras distinto, las métricas cambian. La **validación cruzada k-fold** resuelve esto:

1. Partir los datos en k pliegues iguales (por ejemplo k=5).
2. Entrenar k veces, cada vez usando uno distinto como test.
3. Promediar las k métricas.

Con k=5 o k=10 la estimación es mucho más estable. El costo es entrenar k veces.

Para **series de tiempo**, NO usar k-fold normal: los folds deben respetar el orden temporal (`TimeSeriesSplit`).

## 5. Comparar 3 modelos · baseline vs alternativas

In [ ]:
modelos = {
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}
for nombre, m in modelos.items():
    scores = cross_val_score(m, X, y, cv=5, scoring="roc_auc")
    print(f"{nombre:25s}  AUC = {scores.mean():.3f} ± {scores.std():.3f}")

## 6. Bias-variance tradeoff · por qué modelos más complejos no siempre mejoran

<img src="https://upload.wikimedia.org/wikipedia/commons/9/9f/Bias_and_variance_contributing_to_total_error.svg" width="480" alt="Bias-variance tradeoff">

El error de un modelo se descompone en tres componentes:

- **Bias** (sesgo): el modelo es demasiado simple para capturar el patrón (underfitting).
- **Varianza**: el modelo se ajusta demasiado a los datos de entrenamiento (overfitting).
- **Ruido irreducible**.

A más complejidad → menos bias, más varianza. A menos complejidad → más bias, menos varianza. El objetivo es el **punto dulce**. Validación cruzada ayuda a encontrarlo.

Síntomas de **overfitting**: métricas excelentes en train, malas en test. La diferencia train-test es el termómetro clave.

## 7. Elegir el modelo final · criterios

Para nuestro proyecto, el modelo final se elige por:

1. **Métrica principal** justificada por el caso (no siempre accuracy).
2. **Estabilidad** en validación cruzada (std baja).
3. **Interpretabilidad** si el stakeholder la necesita.
4. **Costo computacional** si el modelo se va a servir.

A veces el mejor modelo es el segundo mejor, porque el primero es una caja negra y el stakeholder necesita explicar por qué se rechazó un préstamo.

---

## 🧑‍🤝‍🧑 Trabajo en grupo sobre el caso

- Evalúen su modelo baseline con 3 métricas apropiadas.
- Comparen contra al menos 2 modelos alternativos con CV.
- Elijan justificadamente el modelo final.

## 📦 Entregable del día

Entrega Semana 3: EDA + modelo final + justificación de métricas.

---

## 📚 Lecturas adicionales

Para profundizar después de la clase, en [`Lecturas.md`](./Lecturas.md) encontrarás una curaduría de artículos técnicos y de negocios en español (4 por día), con copia local en la subcarpeta [`lecturas/`](./lecturas/) cuando la fuente lo permite.

### 🎬 Video recomendado

<iframe width="720" height="405" src="https://www.youtube.com/embed/nFI0eafs0uc" title="Error de generalización: overfitting, bias y varianza" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

**[Error de generalización: overfitting, bias y varianza](https://www.youtube.com/watch?v=nFI0eafs0uc)** · Instituto Humai

_Explica visualmente cómo overfitting y el trade-off bias-variance afectan el desempeño en test._